In [ ]:
%pip install pypdf transformers
dbutils.library.restartPython()

# Ask Jorge — Document Ingestion

Chunks PDF files and writes them to `workspace.default.jorge_cv_chunks` (Delta + CDF).

**Parameters** (job widgets or manual):
- `file_path`: full path to the file in the Databricks Volume
- `full_reindex`: `'true'` to truncate and re-index, `'false'` to append

In [ ]:
import os
import pypdf
from typing import List
from transformers import AutoTokenizer

TABLE_NAME = "workspace.default.jorge_cv_chunks"
TOKENIZER_CACHE = "/tmp/hf_cache"
MAX_TOKENS_PER_CHUNK = 500
SEPARATORS = ["\n\n", "\n", " ", ""]

In [ ]:
dbutils.widgets.text("file_path", "")
dbutils.widgets.text("full_reindex", "true")
file_path = dbutils.widgets.get("file_path")
full_reindex = dbutils.widgets.get("full_reindex").lower() == "true"

print(f"file_path:    {file_path}")
print(f"full_reindex: {full_reindex}")

In [ ]:
def get_pdf_text(pdf_path: str) -> str:
    """Extract all text from a PDF file."""
    text = ""
    try:
        with open(pdf_path, "rb") as f:
            reader = pypdf.PdfReader(f)
            for page in reader.pages:
                extracted = page.extract_text()
                if extracted:
                    text += extracted
    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")
    return text


def chunk_text(text: str, tokenizer) -> List[str]:
    """Recursively split text into chunks under MAX_TOKENS_PER_CHUNK."""
    chunks = [text]
    for separator in SEPARATORS:
        new_chunks = []
        for chunk in chunks:
            if len(tokenizer.encode(chunk)) > MAX_TOKENS_PER_CHUNK:
                parts = chunk.split(separator)
                current = ""
                for part in parts:
                    combined = (current + separator + part) if current else part
                    if len(tokenizer.encode(combined)) <= MAX_TOKENS_PER_CHUNK:
                        current = combined
                    else:
                        if current:
                            new_chunks.append(current.strip())
                        while len(tokenizer.encode(part)) > MAX_TOKENS_PER_CHUNK:
                            tokens = tokenizer.encode(part)
                            new_chunks.append(tokenizer.decode(tokens[:MAX_TOKENS_PER_CHUNK]).strip())
                            part = tokenizer.decode(tokens[MAX_TOKENS_PER_CHUNK:])
                        current = part.strip()
                if current:
                    new_chunks.append(current.strip())
            else:
                new_chunks.append(chunk.strip())
        chunks = new_chunks
    return [c for c in chunks if c]


def ensure_table():
    """Create table if it doesn't exist; truncate if full_reindex=True."""
    try:
        spark.sql(f"DESCRIBE TABLE {TABLE_NAME}")
        if full_reindex:
            print(f"full_reindex=True: truncating {TABLE_NAME}")
            spark.sql(f"TRUNCATE TABLE {TABLE_NAME}")
        spark.sql(
            f"ALTER TABLE {TABLE_NAME} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    except Exception:
        print(f"Table {TABLE_NAME} not found, creating...")
        spark.sql(f"""
            CREATE TABLE {TABLE_NAME} (
                id BIGINT GENERATED BY DEFAULT AS IDENTITY,
                chunk_text STRING,
                source_file STRING
            )
            TBLPROPERTIES (delta.enableChangeDataFeed = true)
        """)
        print(f"Table {TABLE_NAME} created.")

In [ ]:
print(f"Starting ingestion: file_path={file_path}, full_reindex={full_reindex}")

tokenizer = AutoTokenizer.from_pretrained(
    "hf-internal-testing/llama-tokenizer",
    cache_dir=TOKENIZER_CACHE,
)

ensure_table()

filename = os.path.basename(file_path)
text = get_pdf_text(file_path)
if not text:
    raise ValueError(f"No text extracted from {file_path}. Check file format and content.")

chunks = chunk_text(text, tokenizer)
print(f"Extracted {len(chunks)} chunks from {filename}")

data = [(chunk, filename) for chunk in chunks]
(
    spark.createDataFrame(data, ["chunk_text", "source_file"])
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(TABLE_NAME)
)

print(f"Done. {len(chunks)} chunks written to {TABLE_NAME}.")